# 🧬 DiabetAI v9 - Physics-Informed XGBoost

**Strategy: The Trojan Horse**
- Feed pre-calculated physics formulas as features
- Enforce monotonic constraints on ALL clinical variables
- Weight training towards high/low BG scenarios

**New Features:**
- `phys_meal_bolus` = carbs / est_icr
- `phys_corr_bolus` = (cgm - 110) / est_isf
- `phys_net_need` = meal + corr - iob

In [ ]:
!pip install polars xgboost matplotlib scikit-learn -U -q

import polars as pl
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import json
import os

from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive
drive.mount('/content/drive')

print(f"✅ XGBoost: {xgb.__version__}")
print(f"✅ Polars: {pl.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
MODEL_PATH = "/content/drive/MyDrive/DiabetesKDS/models"

TRAIN_FILE = f"{BASE_PATH}/processed/train_ready_balanced_v7.parquet"
TEST_FILE = f"{BASE_PATH}/processed/test_ready_real_world_v7.parquet"

TARGET = 'bolus'

# Target glucose for correction calculation
TARGET_BG = 110.0

print(f"📂 Train: {TRAIN_FILE}")
print(f"📂 Test: {TEST_FILE}")

## Step 1: Load Data & Physics Feature Engineering

In [ ]:
print("="*70)
print("📂 LOADING DATA")
print("="*70)

df_train = pl.read_parquet(TRAIN_FILE)
df_test = pl.read_parquet(TEST_FILE)

print(f"\n✅ Train: {len(df_train):,} rows, {len(df_train.columns)} columns")
print(f"✅ Test: {len(df_test):,} rows, {len(df_test.columns)} columns")

In [ ]:
def add_physics_features(df: pl.DataFrame) -> pl.DataFrame:
    """
    THE TROJAN HORSE: Pre-calculate physics formulas as features.
    Instead of hoping the model learns division, we give it the answer.
    """
    df = df.with_columns([
        # Meal Bolus: carbs / ICR
        (pl.col('carbs') / pl.col('est_icr').clip(lower_bound=1.0))
            .fill_nan(0.0).fill_null(0.0)
            .alias('phys_meal_bolus'),
        
        # Correction Bolus: (CGM - target) / ISF
        # Only positive matters for correction
        ((pl.col('CGM') - TARGET_BG) / pl.col('est_isf').clip(lower_bound=1.0))
            .clip(lower_bound=0.0)
            .fill_nan(0.0).fill_null(0.0)
            .alias('phys_corr_bolus'),
    ])
    
    # Net Need: meal + correction - IOB
    df = df.with_columns([
        (pl.col('phys_meal_bolus') + pl.col('phys_corr_bolus') - pl.col('iob'))
            .clip(lower_bound=0.0)
            .alias('phys_net_need')
    ])
    
    return df

print("\n🧬 Adding Physics Features...")
df_train = add_physics_features(df_train)
df_test = add_physics_features(df_test)

print("\n📊 New Physics Columns:")
print(df_train.select(['carbs', 'CGM', 'iob', 'phys_meal_bolus', 'phys_corr_bolus', 'phys_net_need']).head(10))

In [ ]:
# v9 FEATURE SET (20 features with physics)
V9_FEATURES = [
    # Core Clinical
    'CGM',              # 0  (+1 constraint)
    'carbs',            # 1  (+1 constraint)
    'iob',              # 2  (-1 constraint)
    
    # Physics Features (The Trojan Horse)
    'phys_meal_bolus',  # 3  (+1 constraint) - CRITICAL
    'phys_corr_bolus',  # 4  (+1 constraint) - CRITICAL
    'phys_net_need',    # 5  (+1 constraint)
    
    # Therapy Settings
    'est_tdd',          # 6
    'est_isf',          # 7  (-1: higher ISF = less dose per unit)
    'est_icr',          # 8  (-1: higher ICR = less dose per carb)
    
    # Trends
    'cgm_delta_5min',   # 9  (+1 constraint)
    'cgm_delta_15min',  # 10 (+1 constraint)
    'cgm_velocity',     # 11 (+1 constraint)
    
    # Basal
    'basal',            # 12
    'basal_delta_5min', # 13
    
    # Time
    'time_sin',         # 14
    'time_cos',         # 15
    'is_weekend',       # 16
    
    # Demographics
    'age',              # 17
    'weight',           # 18
    'gender_encoded'    # 19
]

print(f"📋 v9 Features: {len(V9_FEATURES)}")
for i, f in enumerate(V9_FEATURES):
    print(f"   {i:2d}: {f}")

## Step 2: Monotonic Constraints (The Safety Net)

In [ ]:
print("="*70)
print("🔒 MONOTONIC CONSTRAINT DEFINITION")
print("="*70)

# Build constraint tuple
# +1 = Positive (feature up -> dose up)
# -1 = Negative (feature up -> dose down)
#  0 = No constraint

CONSTRAINTS = {
    # Core Clinical
    'CGM': 1,               # Higher glucose -> more dose
    'carbs': 1,             # More carbs -> more dose
    'iob': -1,              # More active insulin -> less dose
    
    # Physics Features (CRITICAL)
    'phys_meal_bolus': 1,   # Pre-calculated meal need
    'phys_corr_bolus': 1,   # Pre-calculated correction need
    'phys_net_need': 1,     # Net insulin need
    
    # Sensitivity
    'est_isf': -1,          # Higher sensitivity -> less dose
    'est_icr': -1,          # Higher ratio -> less dose per carb
    
    # Trends (Rising = need more)
    'cgm_delta_5min': 1,
    'cgm_delta_15min': 1,
    'cgm_velocity': 1,
}

# Build tuple in feature order
monotone_tuple = tuple(
    CONSTRAINTS.get(f, 0) for f in V9_FEATURES
)

print("\n📋 Constraint Map:")
for i, (feat, constraint) in enumerate(zip(V9_FEATURES, monotone_tuple)):
    if constraint != 0:
        symbol = "⬆️ +1" if constraint == 1 else "⬇️ -1"
        print(f"   {i:2d}: {feat:<20} → {symbol}")

constrained_count = sum(1 for c in monotone_tuple if c != 0)
print(f"\n✅ Total Constrained Features: {constrained_count}/{len(V9_FEATURES)}")

## Step 3: Sample Weighting (Focus on High/Low BG)

In [ ]:
print("="*70)
print("⚖️ SAMPLE WEIGHT CALCULATION")
print("="*70)

def calculate_sample_weights(df: pl.DataFrame) -> np.ndarray:
    """
    Weight samples by clinical importance:
    - High BG (>180): Weight 3.0 - Critical corrections
    - Low BG (<70): Weight 5.0 - Hypo prevention crucial
    - Normal: Weight 1.0
    """
    cgm = df.select('CGM').to_numpy().ravel()
    
    weights = np.ones(len(cgm))
    weights[cgm > 180] = 3.0  # High BG - need correction
    weights[cgm < 70] = 5.0   # Hypo - most critical
    
    return weights

train_weights = calculate_sample_weights(df_train)

print(f"\n📊 Weight Distribution:")
print(f"   Weight 1.0 (Normal): {(train_weights == 1.0).sum():,} samples")
print(f"   Weight 3.0 (High BG): {(train_weights == 3.0).sum():,} samples")
print(f"   Weight 5.0 (Low BG): {(train_weights == 5.0).sum():,} samples")
print(f"\n   Total: {len(train_weights):,}")

In [ ]:
# Prepare feature matrices
print("\n🔧 Preparing Feature Matrices...")

# Check all features exist
missing = [f for f in V9_FEATURES if f not in df_train.columns]
if missing:
    raise ValueError(f"❌ Missing features: {missing}")

X_train = df_train.select(V9_FEATURES).to_numpy().astype(np.float32)
y_train = df_train.select(TARGET).to_numpy().ravel().astype(np.float32)

X_test = df_test.select(V9_FEATURES).to_numpy().astype(np.float32)
y_test = df_test.select(TARGET).to_numpy().ravel().astype(np.float32)

print(f"\n📊 Train: X={X_train.shape}, y={y_train.shape}")
print(f"📊 Test:  X={X_test.shape}, y={y_test.shape}")

# Replace inf/nan
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

## Step 4: Train Physics-Aware XGBoost

In [ ]:
print("="*70)
print("🚀 TRAINING XGBOOST v9 (PHYSICS-AWARE)")
print("="*70)

# Check GPU
try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    device = 'cuda' if result.returncode == 0 else 'cpu'
    print(f"\n🖥️ Device: {device.upper()}")
except:
    device = 'cpu'

params = {
    'n_estimators': 2500,
    'learning_rate': 0.015,
    'max_depth': 7,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': device,
    'random_state': 42,
    'monotone_constraints': monotone_tuple,
    'early_stopping_rounds': 75
}

print(f"\n📋 Parameters:")
for k, v in params.items():
    if k != 'monotone_constraints':
        print(f"   {k}: {v}")
print(f"   monotone_constraints: {constrained_count} features locked")

In [ ]:
# Train!
print("\n🏋️ Training with Sample Weights...")

model = xgb.XGBRegressor(**params)

model.fit(
    X_train, y_train,
    sample_weight=train_weights,  # Weighted training!
    eval_set=[(X_test, y_test)],
    verbose=100
)

print(f"\n✅ Training Complete!")
print(f"   Best Iteration: {model.best_iteration}")
print(f"   Best Score: {model.best_score:.6f}")

## Step 5: The 4-Scenario Stress Test

In [ ]:
print("="*70)
print("🧪 4-SCENARIO STRESS TEST")
print("="*70)

def create_test_input(cgm, carbs, iob, est_isf=30, est_icr=10, velocity=0):
    """Create a test input vector with physics features."""
    # Physics calculations
    phys_meal = carbs / est_icr if est_icr > 0 else 0
    phys_corr = max(0, (cgm - TARGET_BG) / est_isf) if est_isf > 0 else 0
    phys_net = max(0, phys_meal + phys_corr - iob)
    
    return np.array([[
        cgm,            # 0: CGM
        carbs,          # 1: carbs
        iob,            # 2: iob
        phys_meal,      # 3: phys_meal_bolus
        phys_corr,      # 4: phys_corr_bolus
        phys_net,       # 5: phys_net_need
        40.0,           # 6: est_tdd
        est_isf,        # 7: est_isf
        est_icr,        # 8: est_icr
        velocity,       # 9: cgm_delta_5min
        velocity * 3,   # 10: cgm_delta_15min
        velocity / 5,   # 11: cgm_velocity
        1.0,            # 12: basal
        0.0,            # 13: basal_delta_5min
        0.0,            # 14: time_sin
        1.0,            # 15: time_cos
        0.0,            # 16: is_weekend
        30.0,           # 17: age
        70.0,           # 18: weight
        0.0             # 19: gender_encoded
    ]], dtype=np.float32)

def calc_math_dose(cgm, carbs, iob, isf=30, icr=10, target=100):
    meal = carbs / icr if icr > 0 and carbs > 0 else 0
    corr = max(0, (cgm - target) / isf) if cgm > target else 0
    return max(0, meal + corr - iob)

# Define scenarios
scenarios = [
    {"name": "A: Meal (Normal BG)", "cgm": 120, "carbs": 50, "iob": 0, "velocity": 0},
    {"name": "B: Correction (High BG)", "cgm": 250, "carbs": 0, "iob": 0, "velocity": 2},
    {"name": "C: Meal + Correction", "cgm": 180, "carbs": 40, "iob": 1, "velocity": 1},
    {"name": "D: Hypo Risk", "cgm": 80, "carbs": 0, "iob": 0, "velocity": -3},
]

print(f"\n{'Scenario':<25} {'Math':>8} {'AI v9':>8} {'Phys_Net':>10} {'Status':>10}")
print("-"*65)

all_pass = True
for s in scenarios:
    X = create_test_input(s['cgm'], s['carbs'], s['iob'], velocity=s['velocity'])
    math_dose = calc_math_dose(s['cgm'], s['carbs'], s['iob'])
    ai_dose = max(0, model.predict(X)[0])
    phys_net = X[0][5]  # phys_net_need
    
    # Evaluate
    if "Hypo" in s['name']:
        status = "✅" if ai_dose < 1.0 else "❌"
    elif "Correction" in s['name']:
        # Should be within 50% of math (more lenient for pure corrections)
        status = "✅" if ai_dose >= math_dose * 0.5 else "❌"
    else:
        # Within 30% of math
        status = "✅" if abs(ai_dose - math_dose) <= math_dose * 0.3 + 0.5 else "❌"
    
    if status == "❌":
        all_pass = False
    
    print(f"{s['name']:<25} {math_dose:>8.2f} {ai_dose:>8.2f} {phys_net:>10.2f} {status:>10}")

print("-"*65)
if all_pass:
    print("🎉 ALL SCENARIOS PASSED!")
else:
    print("⚠️ Some scenarios need review")

In [ ]:
# Detailed comparison: Scenario A vs B
print("\n" + "="*70)
print("🔍 A vs B COMPARISON (Should be DIFFERENT!)")
print("="*70)

X_A = create_test_input(cgm=120, carbs=50, iob=0)
X_B = create_test_input(cgm=250, carbs=0, iob=0)

pred_A = model.predict(X_A)[0]
pred_B = model.predict(X_B)[0]

print(f"\nScenario A (120 mg/dL, 50g carbs):")
print(f"   phys_meal_bolus: {X_A[0][3]:.2f}")
print(f"   phys_corr_bolus: {X_A[0][4]:.2f}")
print(f"   phys_net_need:   {X_A[0][5]:.2f}")
print(f"   AI Prediction:   {pred_A:.2f}U")

print(f"\nScenario B (250 mg/dL, 0g carbs):")
print(f"   phys_meal_bolus: {X_B[0][3]:.2f}")
print(f"   phys_corr_bolus: {X_B[0][4]:.2f}")
print(f"   phys_net_need:   {X_B[0][5]:.2f}")
print(f"   AI Prediction:   {pred_B:.2f}U")

if abs(pred_A - pred_B) > 1.0:
    print(f"\n✅ A and B are DIFFERENT by {abs(pred_A - pred_B):.2f}U!")
else:
    print(f"\n⚠️ A and B too similar: only {abs(pred_A - pred_B):.2f}U difference")

## Step 6: Feature Importance Analysis

In [ ]:
print("="*70)
print("📊 FEATURE IMPORTANCE")
print("="*70)

importance = model.feature_importances_
sorted_idx = np.argsort(importance)[::-1]

print("\n🏆 Top 10 Features:")
for rank, idx in enumerate(sorted_idx[:10], 1):
    feat = V9_FEATURES[idx]
    imp = importance[idx]
    constraint = CONSTRAINTS.get(feat, 0)
    c_str = f"(+{constraint})" if constraint > 0 else (f"({constraint})" if constraint < 0 else "")
    print(f"   {rank:2d}. {feat:<20} {imp:.4f} {c_str}")

# Check physics features rank
print("\n🧬 Physics Feature Rankings:")
physics_feats = ['phys_meal_bolus', 'phys_corr_bolus', 'phys_net_need']
for feat in physics_feats:
    idx = V9_FEATURES.index(feat)
    rank = np.where(sorted_idx == idx)[0][0] + 1
    print(f"   {feat}: Rank #{rank}")

In [ ]:
# Plot importance
plt.figure(figsize=(12, 8))

top_n = 15
top_features = [V9_FEATURES[i] for i in sorted_idx[:top_n]]
top_importance = importance[sorted_idx[:top_n]]

# Color by type
colors = []
for f in top_features:
    if 'phys_' in f:
        colors.append('#ff4b4b')  # Physics = Red
    elif f in ['carbs', 'CGM', 'iob']:
        colors.append('#667eea')  # Core = Purple
    else:
        colors.append('#38ef7d')  # Other = Green

plt.barh(range(top_n), top_importance[::-1], color=colors[::-1])
plt.yticks(range(top_n), top_features[::-1])
plt.xlabel('Importance')
plt.title('v9 Feature Importance\n🔴 Physics | 🟣 Core Clinical | 🟢 Other')
plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/v9_feature_importance.png", dpi=150)
plt.show()

## Step 7: Save Model

In [ ]:
print("="*70)
print("💾 SAVING v9 MODEL")
print("="*70)

os.makedirs(MODEL_PATH, exist_ok=True)

# Save model
model_file = f"{MODEL_PATH}/xgboost_clinical_v9.json"
model.save_model(model_file)
print(f"\n✅ Model: {model_file}")

# Save feature config
config = {
    'version': 'v9',
    'features': V9_FEATURES,
    'target': TARGET,
    'physics_features': ['phys_meal_bolus', 'phys_corr_bolus', 'phys_net_need'],
    'monotone_constraints': list(monotone_tuple),
    'constraint_map': CONSTRAINTS,
    'target_bg': TARGET_BG,
    'best_iteration': model.best_iteration,
    'best_score': float(model.best_score)
}

config_file = f"{MODEL_PATH}/model_config_v9.json"
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Config: {config_file}")

In [ ]:
# Final evaluation
y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, 0, 25)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*70)
print("🎉 v9 TRAINING COMPLETE")
print("="*70)
print(f"\n📊 Test Metrics:")
print(f"   RMSE: {rmse:.4f}")
print(f"   MAE:  {mae:.4f}")
print(f"\n🧬 Physics Features: 3")
print(f"🔒 Constrained: {constrained_count}")
print(f"⚖️ Sample Weights: Active")
print(f"\n🚀 Ready for app.py integration!")